In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src" / "tensor_engine").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "src" / "tensor_engine").is_dir():
    raise RuntimeError("Abre el notebook desde TensorEngine o TensorEngine/notebooks.")
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from tensor_engine import (
    FunctionSpec, LagrangianSourceSpec, TensorEngine,
    WolframXActBridge, spatially_flat_flrw_ansatz, expr_to_latex,
)

In [2]:
source = LagrangianSourceSpec(
    name="notebook_scalar_test",
    expression="R - X/2 - V(phi)",
    functions=(FunctionSpec("V", 1),),
)
model = source.compile()
print("Modelo compilado:", model.name)
print("Fingerprint de fuente:", source.fingerprint)

Modelo compilado: notebook_scalar_test
Fingerprint de fuente: 7dbd28ab63ef433f74addcc0d4f08da94c03931a33e443354e78c2deba81d260


In [3]:
run = TensorEngine().run(
    model,
    ansatz=spatially_flat_flrw_ansatz(),
    output_root=PROJECT_ROOT / "outputs" / "notebook_quickstart",
    wolfram_bridge=WolframXActBridge(timeout_seconds=180),
)
print("Estado:", run.status.value)
print("Verificaciones:", run.package.verification.summary)
print("Run ID:", run.package.run_id)

Estado: success
Verificaciones: {'passed': 57, 'failed': 0, 'undetermined': 0}
Run ID: run_b4caeec143fc0bb15324


In [4]:
from IPython.display import Markdown, display

assert run.status.value == "success"
assert run.package.verification.summary["failed"] == 0
assert run.package.verification.summary["undetermined"] == 0
display(Markdown(
    "### Resultado mínimo\n"
    + f"- $L = {expr_to_latex(run.package.normalized_lagrangian)}$\n"
    + f"- $E_\\phi = {expr_to_latex(run.package.euler.scalar_euler)}$\n"
    + f"- Componentes métricas independientes: {len(run.package.components.independent_metric)}\n"
    + f"- Bundle: `{run.export_bundle.output_directory}`"
))

### Resultado mínimo
- $L = -1\,V\!\left(\phi\right) - \frac{1}{2}\,u{}_{d0}\,g{}^{d0}{}^{d1}\,u{}_{d1} + R{}_{d0}{}_{d1}{}_{d2}{}_{d3}\,g{}^{d0}{}^{d2}\,g{}^{d1}{}^{d3}$
- $E_\phi = -1\,V^{(1)}\!\left(\phi\right) + \nabla_{d0}\!\left(u{}^{d0}\right)$
- Componentes métricas independientes: 10
- Bundle: `C:\Investigacion\TensorEngine\outputs\notebook_quickstart\notebook-scalar-test-b4caeec143fc`